# Logo Recognition Clustring

This project aims to address the challenge of logo recognition and similarity search in the context of limited available datasets. Due to the vast diversity of logos and the constant emergence of new designs, traditional logo recognition systems often struggle to maintain high accuracy and robustness. To tackle this issue, the project proposes the use of synthetic data generation techniques alongside advanced deep learning models to significantly improve the system's ability to accurately identify and find similar logos.


**Notebooks:**

[Prompts Embedding](https://colab.research.google.com/drive/19DhkXGYnHh2o9k_8Iac1eFk8YI_J58Vd?usp=sharing)

[Logo Images Embedding](https://colab.research.google.com/drive/1fZhDERCptm1eYns_vpufeIYWOPCgAOyg?usp=sharing)




## Imports

In [ ]:
# This get the RAPIDS-Colab install files and test check your GPU.  Run this and the next cell only.
# Please read the output of this cell.  If your Colab Instance is not RAPIDS compatible, it will warn you and give you remediation steps.
!git clone https://github.com/rapidsai/rapidsai-csp-utils.git
!python rapidsai-csp-utils/colab/pip-install.py


Cloning into 'rapidsai-csp-utils'...
remote: Enumerating objects: 586, done.
remote: Counting objects: 100% (152/152), done.
remote: Compressing objects: 100% (70/70), done.
remote: Total 586 (delta 122), reused 82 (delta 82), pack-reused 434 (from 3)
Receiving objects: 100% (586/586), 191.99 KiB | 5.19 MiB/s, done.
Resolving deltas: 100% (296/296), done.
Installing RAPIDS remaining 24.12.* libraries
Using Python 3.11.11 environment at: /usr
Resolved 154 packages in 2.25s
 Downloaded ucx-py-cu12
 Downloaded libkvikio-cu12
 Downloaded rmm-cu12
 Downloaded cuspatial-cu12
 Downloaded dask
 Downloaded datashader
 Downloaded cuda-python
 Downloaded libucx-cu12
 Downloaded cucim-cu12
 Downloaded libcuspatial-cu12
 Downloaded pylibraft-cu12
 Downloaded scikit-image
 Downloaded nvidia-nvcomp-cu12
 Downloaded cudf-cu12
 Downloaded pylibcudf-cu12
 Downloaded raft-dask-cu12
 Downloaded libcudf-cu12
 Downloaded cuml-cu12
 Downloaded cuvs-cu12
 Downloaded cugraph-cu12
 Downloaded pylibcugraph-cu12


In [ ]:
import cudf
import cuml

(
    cudf.__version__,
    cuml.__version__,

 )

('24.12.00', '24.12.00')

In [ ]:
!pip install datasets -q

import os
import numpy as np
import pandas as pd
from tqdm import tqdm
tqdm.pandas()
import torch
import datasets
import warnings
warnings.filterwarnings('ignore')


output_dir = f"/content/drive/MyDrive/logo_recognition_similarity_search_project/output"
os.makedirs(output_dir,exist_ok=True)
ds_dir = f"{output_dir}/logo-recognition-embeddings"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 485.4/485.4 kB 9.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 10.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.5/143.5 kB 14.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.8/194.8 kB 19.2 MB/s eta 0:00:00


## Load Dataset

In [ ]:
ds = datasets.load_from_disk(ds_dir)
ds = ds.with_format("numpy")
ds

Loading dataset from disk:   0%|          | 0/36 [00:00<?, ?it/s]

Dataset({
    features: ['id', 'path', 'prompt', 'prompt_embedding', 'image_embedding'],
    num_rows: 1777584
})

In [ ]:
ds.features

{'id': Value(dtype='string', id=None),
 'path': Value(dtype='string', id=None),
 'prompt': Value(dtype='string', id=None),
 'prompt_embedding': Sequence(feature=Value(dtype='float32', id=None), length=-1, id=None),
 'image_embedding': Sequence(feature=Value(dtype='float32', id=None), length=-1, id=None)}

In [ ]:
# Extract image and text embeddings
image_embeddings = np.array(ds['image_embedding'])
prompt_embeddings = np.array(ds['prompt_embedding'])

# Concatenate image and text embeddings
combined_embeddings = np.hstack((image_embeddings, prompt_embeddings))

# # Extract text prompts
# prompts = ds['prompt']
del image_embeddings
del prompt_embeddings

combined_embeddings.shape

(1777584, 2432)

## Embedding

In [ ]:
import torch
import torch.nn as nn
from sentence_transformers import SentenceTransformer
from torchvision import models, transforms
from PIL import Image
import pandas as pd

class EmbeddingModel(nn.Module):
    def __init__(self, device="cuda" if torch.cuda.is_available() else "cpu"):
        """
        Defines an `EmbeddingModel` that extracts text and image embeddings
        and concatenates them into a `combined_embedding`.
        """
        super(EmbeddingModel, self).__init__()
        self.device = device

        # Load SentenceTransformer for text embeddings
        self.text_model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2',device=self.device)

        # Load ResNet-50 for image embeddings and remove the final classification layer
        resnet = models.resnet50(pretrained=True)
        self.image_model = nn.Sequential(*list(resnet.children())[:-1])
        self.image_model.to(self.device)
        self.image_model.eval()

        # Define image transformations before feeding into ResNet-50
        self.transform = transforms.Compose([
            transforms.Resize((128, 128)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        ])

    def forward_text(self, prompt):
        """ Extracts `text_embedding` using SentenceTransformer """
        return torch.tensor(self.text_model.encode(prompt)).to(self.device)

    def forward_image(self, image_path):
        """ Extracts `image_embedding` using ResNet-50 """
        image = Image.open(image_path).convert("RGB")
        image = self.transform(image).unsqueeze(0).to(self.device)

        with torch.no_grad():
            embedding = self.image_model(image)

        return embedding.view(-1).to(self.device)

    def forward(self, prompt, image_path):
        """ Generates `combined_embedding` by concatenating text and image embeddings """
        text_emb = self.forward_text(prompt)
        image_emb = self.forward_image(image_path)
        combined_emb = torch.cat((text_emb, image_emb), dim=0)
        return combined_emb


In [ ]:
# Enable CUDA if available
device = "cuda" if torch.cuda.is_available() else "cpu"

# Instantiate the model and move it to the appropriate device
model = EmbeddingModel(device=device)

# Sample prompt and image
prompt = "A minimalist logo for a finance company"
image_path = f"{output_dir}/logo_image.png"

# Run the model to extract embeddings
combined_embedding = model(prompt, image_path)

print("Combined Embedding Size:", combined_embedding.shape)


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth
100%|██████████| 97.8M/97.8M [00:00<00:00, 131MB/s]


Combined Embedding Size: torch.Size([2432])


## Clustring

### UMAP

This section applies **UMAP** (Uniform Manifold Approximation and Projection) to the combined image and text embeddings. **UMAP** is a dimensionality reduction technique well-suited for visualizing high-dimensional data while preserving the underlying data structure.  The reduced embeddings are then used for clustering.  The code in this section assumes the existence of a preprocessed dataset with combined embeddings as generated in earlier parts of the notebooks.

In [ ]:
from cuml.manifold import UMAP

umap_model = UMAP(n_components=200,
                  metric='cosine',
                  min_dist=0.0,
                  random_state=42,
                  verbose=True
                  )

In [ ]:
reduced_embeddings = umap_model.fit_transform(combined_embeddings)

[D] [22:40:43.099256] /__w/cuml/cuml/cpp/src/umap/runner.cuh:107 n_neighbors=15
[D] [22:40:43.253868] /__w/cuml/cuml/cpp/src/umap/runner.cuh:129 Calling knn graph run
[D] [22:54:57.670959] /__w/cuml/cuml/cpp/src/umap/runner.cuh:135 Done. Calling fuzzy simplicial set
[D] [22:54:57.743349] /__w/cuml/cuml/cpp/src/umap/fuzzy_simpl_set/naive.cuh:318 Smooth kNN Distances
[D] [22:54:57.743808] /__w/cuml/cuml/cpp/src/umap/fuzzy_simpl_set/naive.cuh:320 sigmas = [ 0.100173, 0.122496, 0.0331979, 0.103231, 0.0130104, 0.136421, 0.108887, 0.0905116, 0.0691073, 0.070961, 0.087851, 0.0876267, 0.120947, 0.116036, 0.166859, 0.132617, 0.0982676, 0.117495, 0.12125, 0.106179, 0.0579612, 0.111126, 0.0264722, 0.0271819, 0.0169317 ]

[D] [22:54:57.743850] /__w/cuml/cuml/cpp/src/umap/fuzzy_simpl_set/naive.cuh:322 rhos = [ 1.13249e-06, 2.98023e-07, 0.0691797, 5.96046e-08, 0.183514, 2.98023e-07, 7.7486e-07, 1.96695e-06, 6.55651e-07, 9.53674e-07, 1.13249e-06, 1.3113e-06, 1.66893e-06, 7.15256e-07, 1.54972e-06, 1.4

In [ ]:
reduced_embeddings.shape

(1777584, 200)

### KMeans

KMeans is a clustering algorithm that aims to partition n observations into k clusters, where each observation belongs to the cluster with the nearest mean (cluster centers or cluster centroid), serving as a prototype of the cluster.  We will use KMeans to cluster the reduced embeddings.


In [ ]:
from cuml.cluster.kmeans import KMeans

num_clusters = 282276

kmeans_model = KMeans(
                      n_clusters=num_clusters,
                      max_iter=300,
                      random_state=42,
                      verbose=True,
                      )


In [ ]:
kmeans_model.fit(reduced_embeddings)

KMeans()

In [ ]:
kmeans_model.labels_.shape

(1777584,)

In [ ]:
unique_labels, counts = np.unique(kmeans_model.labels_, return_counts=True)
print(f"Number of unique labels: {len(unique_labels)}")
label_counts_df = pd.DataFrame({'Label': unique_labels, 'Count': counts})
label_counts_df

Number of unique labels: 282276


,Label,Count
0,0,7
1,1,4
2,2,1
3,3,7
4,4,1
...,...,...
282271,282271,1
282272,282272,92
282273,282273,5
282274,282274,2


In [ ]:
label_counts_df.describe()

,Label,Count
count,282276.00000,282276.000000
mean,141137.50000,6.297326
std,81486.20663,5.891349
min,0.00000,1.000000
25%,70568.75000,2.000000
50%,141137.50000,5.000000
75%,211706.25000,9.000000
max,282275.00000,216.000000


## Save

This section saves the clustering results (labels and soft clusters) to google drive.  These files will be used in subsequent notebooks for analysis and visualization.  The soft clusters represent the probability of each data point belonging to each cluster, providing a measure of uncertainty in the cluster assignments.

In [ ]:
np.save(f"{output_dir}/reduced_embeddings.npy", reduced_embeddings)
np.save(f"{output_dir}/kmeans_labels.npy", kmeans_model.labels_)

In [ ]:
from joblib import dump

dump(kmeans_model, f"{output_dir}/kmeans_model.joblib")

In [ ]:
ds = ds.add_column('category', kmeans_model.labels_.tolist())

In [ ]:
ds

Dataset({
    features: ['id', 'path', 'prompt', 'prompt_embedding', 'image_embedding', 'category'],
    num_rows: 1777584
})

In [ ]:
ds.features

{'id': Value(dtype='string', id=None),
 'path': Value(dtype='string', id=None),
 'prompt': Value(dtype='string', id=None),
 'prompt_embedding': Sequence(feature=Value(dtype='float32', id=None), length=-1, id=None),
 'image_embedding': Sequence(feature=Value(dtype='float32', id=None), length=-1, id=None),
 'category': Value(dtype='int64', id=None)}

In [ ]:
ds.save_to_disk(f"{output_dir}/logo-recognition-embeddings-with-category")

Saving the dataset (0/36 shards):   0%|          | 0/1777584 [00:00<?, ? examples/s]

In [ ]:
df = ds.remove_columns(['image_embedding', 'prompt_embedding']).to_pandas()
df.head()

,id,path,prompt,category
0,1104391803911815278,images/00/00/1104391803911815278_0_0.png,cute simple baby shark logo for company,74058
1,1104391803911815278,images/00/00/1104391803911815278_0_1.png,cute simple baby shark logo for company,74058
2,1104391803911815278,images/00/00/1104391803911815278_1_0.png,cute simple baby shark logo for company,74058
3,1104391803911815278,images/00/00/1104391803911815278_1_1.png,cute simple baby shark logo for company,74058
4,1132464139621646377,images/00/00/1132464139621646377_0_0.png,"food truck, sells french fries, logo on side i...",155635


In [ ]:
df.shape

(1777584, 4)

In [ ]:
df.to_csv(f"{output_dir}/logo-recognition-with-category.tsv", sep='\t', index=False)

In [ ]:
df = pd.read_csv(f"{output_dir}/logo-recognition-with-category.tsv", sep='\t')
df.head()

,id,path,prompt,category
0,1104391803911815278,images/00/00/1104391803911815278_0_0.png,cute simple baby shark logo for company,74058
1,1104391803911815278,images/00/00/1104391803911815278_0_1.png,cute simple baby shark logo for company,74058
2,1104391803911815278,images/00/00/1104391803911815278_1_0.png,cute simple baby shark logo for company,74058
3,1104391803911815278,images/00/00/1104391803911815278_1_1.png,cute simple baby shark logo for company,74058
4,1132464139621646377,images/00/00/1132464139621646377_0_0.png,"food truck, sells french fries, logo on side i...",155635
